# BiLSTM-CRF for Named Entity Recognition
## With Hand-Implemented CRF Forward Algorithm & Viterbi Decoding

---

## 📌 Why BiLSTM-CRF?

NER (Named Entity Recognition) is a sequence labeling task: given a sentence, assign a tag to each token.

```
"Barack  Obama  visited  Paris  yesterday"
  B-PER   I-PER    O      B-LOC     O
```

### The problem with a plain BiLSTM classifier
A naive approach: pass each token through a BiLSTM → linear layer → softmax → pick the highest-scoring tag independently.

**Problem**: it can produce *invalid* tag sequences like `I-PER` at the start (no `B-PER` before it), or `B-PER I-LOC` (entity type mismatch). The model has no notion of tag-to-tag constraints.

### The CRF fix
A **Conditional Random Field (CRF)** layer sits on top of the BiLSTM and scores entire sequences, not individual tags. It learns a **transition matrix** `T[i,j]` = score of going from tag `i` to tag `j`. At inference time, we find the globally best tag sequence using **Viterbi** — a dynamic programming algorithm.

---

## 🏗️ Architecture Overview

```
Input tokens
     │
     ▼
Embedding Layer          (token → dense vector)
     │
     ▼
BiLSTM                   (context from both directions)
     │
     ▼
Linear Projection        (hidden → num_tags emission scores)
     │
     ▼
CRF Layer
  ├─ Training : Forward Algorithm  → log-partition Z
  │             NLL = Z − score(gold sequence)
  └─ Inference: Viterbi Algorithm  → best tag sequence
```

---

## 🧮 Mathematical Foundation

### Emission scores
The BiLSTM+Linear gives us `E[t, k]` = score that token at position `t` has tag `k`.

### Sequence score
For a tag sequence `y = (y₀, y₁, …, y_{T-1})`:

```
score(y) = Σ_t E[t, y_t]  +  Σ_t T[y_{t-1}, y_t]  +  T_start[y_0]  +  T_end[y_{T-1}]
```

- **Emission terms**: how well the BiLSTM thinks tag `y_t` fits token `t`
- **Transition terms**: how valid tag `y_{t-1} → y_t` is (learned constraint)
- **Start/End terms**: special rows in the transition matrix for sequence boundaries

### Training objective — Negative Log-Likelihood

```
loss = -log P(y* | x)  =  -[ score(y*)  −  log Σ_{all y} exp(score(y)) ]
                                                 └────────── Z ──────────┘
```

`Z` is the **log-partition function** — the log of the sum of exponentiated scores over ALL possible sequences. Computing it naively is exponential. The **Forward Algorithm** computes it in O(T × K²) using dynamic programming.

### Forward Algorithm (log-space)
```
α[0, k] = T_start[k] + E[0, k]          # initialise
α[t, k] = logsumexp_j( α[t-1, j] + T[j,k] ) + E[t, k]
Z = logsumexp_k( α[T-1, k] + T_end[k] )
```

### Viterbi Algorithm (inference)
Same recurrence but replace `logsumexp` → `max`, and store back-pointers:
```
δ[0, k] = T_start[k] + E[0, k]
δ[t, k] = max_j( δ[t-1, j] + T[j,k] ) + E[t, k]
best_path = backtrack pointers from argmax at T-1
```

---
## 1. Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from typing import List, Tuple, Dict
import random, numpy as np

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## 2. Toy NER Dataset

We use the **BIO** (Begin–Inside–Outside) tagging scheme:
- `B-PER` / `I-PER` — person name begins / continues
- `B-LOC` / `I-LOC` — location begins / continues  
- `B-ORG` / `I-ORG` — organisation begins / continues
- `O` — outside any entity

**BIO constraint**: `I-X` can only follow `B-X` or `I-X`. The CRF learns this automatically from data — we don't hard-code it.

In [ ]:
# ── Toy sentences (tokens, tags) ──────────────────────────────────────────────
RAW_DATA = [
    (["Barack", "Obama", "visited", "Berlin", "yesterday"],
     ["B-PER", "I-PER", "O", "B-LOC", "O"]),
    (["Google", "was", "founded", "in", "California"],
     ["B-ORG", "O", "O", "O", "B-LOC"]),
    (["The", "president", "met", "Angela", "Merkel", "in", "Paris"],
     ["O", "O", "O", "B-PER", "I-PER", "O", "B-LOC"]),
    (["Apple", "Inc", "opened", "a", "store", "in", "London"],
     ["B-ORG", "I-ORG", "O", "O", "O", "O", "B-LOC"]),
    (["Elon", "Musk", "leads", "Tesla", "and", "SpaceX"],
     ["B-PER", "I-PER", "O", "B-ORG", "O", "B-ORG"]),
    (["Paris", "is", "the", "capital", "of", "France"],
     ["B-LOC", "O", "O", "O", "O", "B-LOC"]),
    (["Microsoft", "acquired", "GitHub", "in", "2018"],
     ["B-ORG", "O", "B-ORG", "O", "O"]),
    (["Albert", "Einstein", "was", "born", "in", "Germany"],
     ["B-PER", "I-PER", "O", "O", "O", "B-LOC"]),
    (["Amazon", "opened", "offices", "in", "New", "York"],
     ["B-ORG", "O", "O", "O", "B-LOC", "I-LOC"]),
    (["Narendra", "Modi", "visited", "Tokyo", "last", "week"],
     ["B-PER", "I-PER", "O", "B-LOC", "O", "O"]),
]

# ── Build vocabularies ────────────────────────────────────────────────────────
PAD, UNK = "<PAD>", "<UNK>"

all_tokens = [tok for sent, _ in RAW_DATA for tok in sent]
all_tags   = [tag for _, tags in RAW_DATA for tag in tags]

word2idx: Dict[str, int] = {PAD: 0, UNK: 1}
for w in sorted(set(all_tokens)):
    word2idx[w] = len(word2idx)

tag2idx: Dict[str, int] = {}
for t in sorted(set(all_tags)):
    tag2idx[t] = len(tag2idx)

# Special start/end tags used only inside the CRF transition matrix
START_TAG, END_TAG = "<START>", "<END>"
tag2idx[START_TAG] = len(tag2idx)
tag2idx[END_TAG]   = len(tag2idx)
idx2tag = {v: k for k, v in tag2idx.items()}

print(f"Vocab size : {len(word2idx)}")
print(f"Tag set    : {tag2idx}")

---
## 3. Dataset & DataLoader

In [ ]:
class NERDataset(Dataset):
    """
    Converts raw (tokens, tags) pairs into integer tensors.
    Each sample is already a full sentence — no padding here;
    we handle variable lengths with a custom collate function.
    """
    def __init__(self, data, word2idx, tag2idx):
        self.samples = []
        for tokens, tags in data:
            x = torch.tensor([word2idx.get(t, word2idx[UNK]) for t in tokens], dtype=torch.long)
            y = torch.tensor([tag2idx[t] for t in tags],                        dtype=torch.long)
            self.samples.append((x, y))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_fn(batch):
    """
    Pad sequences in a batch to the same length.
    Returns:
        x_pad   : (B, T_max)  — padded token indices
        y_pad   : (B, T_max)  — padded tag indices
        lengths : (B,)        — actual lengths (needed for pack_padded_sequence)
    """
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    x_pad   = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=word2idx[PAD])
    y_pad   = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=tag2idx['O'])  # pad tag irrelevant
    return x_pad, y_pad, lengths


dataset    = NERDataset(RAW_DATA, word2idx, tag2idx)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
print(f"Dataset size: {len(dataset)} sentences")

---
## 4. CRF Layer — Forward Algorithm & Viterbi

This is the **core of the notebook**. We implement two algorithms inside the CRF:

### 4.1 `_score_sentence` — gold sequence score
Score a *known* gold tag sequence: sum emission + transition terms.

### 4.2 `_forward_alg` — log-partition function Z
Compute log Σ exp(score(y)) over all sequences using DP.

```
For each timestep t:
  for each tag k:
    α[t,k] = log Σ_j exp( α[t-1,j] + T[j,k] ) + E[t,k]
              └────────── logsumexp over prev tags ─────────┘
```

We use the **log-sum-exp trick** for numerical stability:
```
logsumexp(a) = max(a) + log Σ exp(a - max(a))
```

### 4.3 `viterbi_decode` — best sequence at inference
Same DP, but `logsumexp → max`. We also store **back-pointers** so we can trace back the best path.

In [ ]:
class CRF(nn.Module):
    """
    Conditional Random Field layer.

    Parameters
    ----------
    num_tags : int   — number of output tags (excluding START/END)
    tag2idx  : dict  — maps tag string → integer index
    """

    def __init__(self, num_tags: int, tag2idx: Dict[str, int]):
        super().__init__()
        self.num_tags = num_tags
        self.start_idx = tag2idx[START_TAG]
        self.end_idx   = tag2idx[END_TAG]

        # ── Transition matrix ──────────────────────────────────────────────────
        # transitions[i, j] = score of moving FROM tag i TO tag j
        # Shape: (num_tags, num_tags)  — includes START and END rows/cols
        # Initialised randomly; learned during training.
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))

        # Hard constraints: nothing transitions TO start, nothing transitions FROM end
        with torch.no_grad():
            self.transitions[:, self.start_idx] = -10000.0   # no tag → START
            self.transitions[self.end_idx, :]   = -10000.0   # END → no tag

    # ──────────────────────────────────────────────────────────────────────────
    # Helper: log-sum-exp over last dimension
    # ──────────────────────────────────────────────────────────────────────────
    @staticmethod
    def _log_sum_exp(vec: torch.Tensor) -> torch.Tensor:
        """
        Numerically stable logsumexp along the last dimension.
        vec : (..., K)
        returns (...,)

        Trick: logsumexp(a) = m + log Σ exp(a − m),  m = max(a)
        Subtracting m keeps exp() arguments non-positive → no overflow.
        """
        m, _ = vec.max(dim=-1, keepdim=True)   # (..., 1)
        return m.squeeze(-1) + (vec - m).exp().sum(dim=-1).log()

    # ──────────────────────────────────────────────────────────────────────────
    # 4.1  Score a gold tag sequence
    # ──────────────────────────────────────────────────────────────────────────
    def _score_sentence(
        self,
        emissions: torch.Tensor,   # (T, num_tags) — BiLSTM output for one sentence
        tags:      torch.Tensor,   # (T,)          — gold tag indices
    ) -> torch.Tensor:             # scalar
        """
        score(y*) = T[START → y*_0] + E[0, y*_0]
                  + Σ_{t=1}^{T-1} ( T[y*_{t-1} → y*_t] + E[t, y*_t] )
                  + T[y*_{T-1} → END]
        """
        T = emissions.size(0)
        score = torch.zeros(1, device=emissions.device)

        # Prepend START tag to iterate transitions naturally
        tags_ext = torch.cat([
            torch.tensor([self.start_idx], device=tags.device), tags
        ])  # length T+1

        for t in range(T):
            # Transition from previous tag to current tag
            trans_score  = self.transitions[tags_ext[t], tags_ext[t + 1]]
            # Emission score at position t for the gold tag
            emit_score   = emissions[t, tags[t]]
            score        = score + trans_score + emit_score

        # Final transition to END
        score = score + self.transitions[tags[-1], self.end_idx]
        return score

    # ──────────────────────────────────────────────────────────────────────────
    # 4.2  Forward Algorithm — compute log Z
    # ──────────────────────────────────────────────────────────────────────────
    def _forward_alg(self, emissions: torch.Tensor) -> torch.Tensor:
        """
        Compute log Σ_{all y} exp(score(y))  in O(T × K²) using DP.

        emissions: (T, num_tags)
        returns  : scalar  (log partition function Z)

        DP state:
          alpha[k] = log Σ_{all partial paths ending at tag k at position t}
                         exp( score of that partial path )

        Recurrence (log-space):
          alpha_new[k] = logsumexp_j( alpha[j] + T[j,k] ) + E[t, k]
        """
        T, K = emissions.size()

        # ── Initialise: start from START tag ──────────────────────────────────
        # alpha[k] = T[START → k] + E[0, k]
        alpha = self.transitions[self.start_idx, :] + emissions[0]   # (K,)

        # ── Recurrence over positions 1 … T-1 ────────────────────────────────
        for t in range(1, T):
            # Broadcast: alpha_expanded[j, k] = alpha[j] + T[j, k]
            # Then logsumexp over j (axis 0) → (K,)
            alpha_expanded = alpha.unsqueeze(1) + self.transitions   # (K, K)
            #                  (K, 1)            (K, K)
            # alpha_expanded[j, k] = alpha[j] + T[j, k]

            alpha = self._log_sum_exp(alpha_expanded.T) + emissions[t]
            # .T makes shape (K, K) where dim-1 is 'j'
            # logsumexp over dim=-1 → (K,)  then add emission

        # ── Finalise: transition to END tag ──────────────────────────────────
        terminal = alpha + self.transitions[:, self.end_idx]   # (K,)
        return self._log_sum_exp(terminal.unsqueeze(0)).squeeze()  # scalar

    # ──────────────────────────────────────────────────────────────────────────
    # Training loss
    # ──────────────────────────────────────────────────────────────────────────
    def neg_log_likelihood(
        self,
        emissions: torch.Tensor,  # (T, num_tags)
        tags:      torch.Tensor,  # (T,)
    ) -> torch.Tensor:
        """
        NLL = log Z  −  score(y*)

        Minimising NLL is equivalent to maximising log P(y* | x).
        """
        log_Z        = self._forward_alg(emissions)
        gold_score   = self._score_sentence(emissions, tags)
        return log_Z - gold_score

    # ──────────────────────────────────────────────────────────────────────────
    # 4.3  Viterbi Decoding
    # ──────────────────────────────────────────────────────────────────────────
    def viterbi_decode(
        self,
        emissions: torch.Tensor,  # (T, num_tags)
    ) -> Tuple[List[int], float]:
        """
        Find the highest-scoring tag sequence using dynamic programming.

        Difference from Forward:
          • logsumexp  →  max
          • We store back-pointers (argmax) at each step

        Returns
        -------
        best_path  : List[int]  — tag index per token
        best_score : float
        """
        T, K = emissions.size()
        backpointers = []   # list of (K,) tensors, one per step

        # ── Initialise ────────────────────────────────────────────────────────
        delta = self.transitions[self.start_idx, :] + emissions[0]  # (K,)

        # ── Recurrence ────────────────────────────────────────────────────────
        for t in range(1, T):
            # delta_expanded[j, k] = delta[j] + T[j, k]
            delta_expanded = delta.unsqueeze(1) + self.transitions   # (K, K)

            # Best previous tag for each current tag k
            best_scores, best_prev = delta_expanded.max(dim=0)       # each (K,)

            backpointers.append(best_prev)   # store argmax
            delta = best_scores + emissions[t]

        # ── Finalise ─────────────────────────────────────────────────────────
        terminal      = delta + self.transitions[:, self.end_idx]  # (K,)
        best_score, best_last_tag = terminal.max(dim=0)

        # ── Backtrack ────────────────────────────────────────────────────────
        best_path = [best_last_tag.item()]
        for bp in reversed(backpointers):
            best_path.append(bp[best_path[-1]].item())
        best_path.reverse()

        return best_path, best_score.item()

---
## 5. BiLSTM-CRF Model

The full model:
1. **Embedding** — maps each token index to a learnable dense vector
2. **BiLSTM** — two-direction LSTM that captures left and right context
3. **Linear** — projects the BiLSTM output to `num_tags` emission scores
4. **CRF** — scores full sequences and decodes with Viterbi

In [ ]:
class BiLSTMCRF(nn.Module):
    """
    BiLSTM-CRF for Named Entity Recognition.

    Architecture
    ------------
    token indices
      → Embedding (vocab_size, embed_dim)
      → Dropout
      → BiLSTM   (embed_dim → 2*hidden_dim)
      → Dropout
      → Linear   (2*hidden_dim → num_tags)  — emission scores
      → CRF      (transition matrix + decoding)
    """

    def __init__(
        self,
        vocab_size:  int,
        embed_dim:   int,
        hidden_dim:  int,
        num_tags:    int,
        tag2idx:     Dict[str, int],
        num_layers:  int = 1,
        dropout:     float = 0.3,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # bidirectional=True → output is 2*hidden_dim per position
        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout   = nn.Dropout(dropout)
        self.hidden2tag = nn.Linear(hidden_dim * 2, num_tags)  # 2 directions
        self.crf        = CRF(num_tags, tag2idx)

    # ── Forward pass → emission scores ───────────────────────────────────────
    def _get_emissions(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        x       : (B, T_max)  padded token indices
        lengths : (B,)        true sequence lengths
        returns : (B, T_max, num_tags)  emission logits
        """
        embeds = self.dropout(self.embedding(x))          # (B, T, E)

        # Pack so LSTM ignores padding
        packed = nn.utils.rnn.pack_padded_sequence(
            embeds, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        lstm_out, _ = self.bilstm(packed)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
        # lstm_out: (B, T, 2*H)

        emissions = self.hidden2tag(self.dropout(lstm_out))  # (B, T, num_tags)
        return emissions

    # ── Training step ─────────────────────────────────────────────────────────
    def forward(
        self,
        x:       torch.Tensor,  # (B, T)
        tags:    torch.Tensor,  # (B, T)
        lengths: torch.Tensor,  # (B,)
    ) -> torch.Tensor:           # scalar loss
        """
        Returns total NLL loss for the batch.
        We process each sentence individually because the CRF
        operates on true (unpadded) lengths.
        """
        emissions = self._get_emissions(x, lengths)  # (B, T, K)
        loss = torch.tensor(0.0, device=x.device, requires_grad=True)

        for i, L in enumerate(lengths):
            # Slice to actual length — no padding confusion
            emit_i = emissions[i, :L]   # (L, K)
            tags_i = tags[i, :L]        # (L,)
            loss = loss + self.crf.neg_log_likelihood(emit_i, tags_i)

        return loss / len(lengths)   # mean over batch

    # ── Inference ─────────────────────────────────────────────────────────────
    def predict(
        self,
        x:       torch.Tensor,  # (B, T)
        lengths: torch.Tensor,  # (B,)
    ) -> List[List[int]]:
        """Return Viterbi-decoded tag sequences for each sentence."""
        self.eval()
        with torch.no_grad():
            emissions = self._get_emissions(x, lengths)
        predictions = []
        for i, L in enumerate(lengths):
            path, _ = self.crf.viterbi_decode(emissions[i, :L])
            predictions.append(path)
        return predictions

---
## 6. Instantiate Model & Optimizer

In [ ]:
VOCAB_SIZE = len(word2idx)
NUM_TAGS   = len(tag2idx)      # includes START and END
EMBED_DIM  = 64
HIDDEN_DIM = 128

model = BiLSTMCRF(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_tags=NUM_TAGS,
    tag2idx=tag2idx,
    num_layers=1,
    dropout=0.3,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

---
## 7. Training Loop

In [ ]:
def train(model, loader, optimizer, scheduler, epochs=80):
    """
    Standard PyTorch training loop.
    Loss printed every 10 epochs.
    """
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0

        for x_batch, y_batch, lengths in loader:
            x_batch  = x_batch.to(device)
            y_batch  = y_batch.to(device)
            lengths  = lengths.to(device)

            optimizer.zero_grad()
            loss = model(x_batch, y_batch, lengths)
            loss.backward()

            # Gradient clipping — important for RNNs
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        history.append(avg_loss)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}/{epochs} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    return history


history = train(model, dataloader, optimizer, scheduler, epochs=80)

---
## 8. Loss Curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(history) + 1), history, linewidth=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('NLL Loss')
plt.title('BiLSTM-CRF Training Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Evaluation on Training Data

In [ ]:
def evaluate(model, dataset, idx2tag):
    correct = total = 0

    for (x, y) in dataset:
        x_in  = x.unsqueeze(0).to(device)   # (1, T)
        lens  = torch.tensor([len(x)]).to(device)
        preds = model.predict(x_in, lens)[0]  # List[int]

        for pred_idx, gold_idx in zip(preds, y.tolist()):
            total   += 1
            correct += int(pred_idx == gold_idx)

    print(f"Token-level accuracy: {correct}/{total} = {correct/total:.2%}")


evaluate(model, dataset, idx2tag)

---
## 10. Qualitative Predictions

In [ ]:
def predict_sentence(model, sentence: List[str], word2idx, idx2tag):
    """
    Predict NER tags for a new sentence (list of tokens).
    Tokens not in vocabulary are mapped to UNK.
    """
    model.eval()
    indices = [word2idx.get(tok, word2idx[UNK]) for tok in sentence]
    x  = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    ln = torch.tensor([len(indices)]).to(device)

    tag_indices = model.predict(x, ln)[0]
    tags = [idx2tag[i] for i in tag_indices]

    # Pretty-print
    print(f"{'Token':<15} {'Predicted Tag':<12}")
    print("-" * 28)
    for tok, tag in zip(sentence, tags):
        marker = "  ←" if tag != "O" else ""
        print(f"{tok:<15} {tag:<12}{marker}")


# ── Test on training sentences ────────────────────────────────────────────────
print("=" * 40)
print("Sentence 1: Barack Obama visited Berlin")
print("=" * 40)
predict_sentence(model, ["Barack", "Obama", "visited", "Berlin", "yesterday"], word2idx, idx2tag)

print()
print("=" * 40)
print("Sentence 2: Apple Inc opened a store in London")
print("=" * 40)
predict_sentence(model, ["Apple", "Inc", "opened", "a", "store", "in", "London"], word2idx, idx2tag)

print()
print("=" * 40)
print("Sentence 3: Elon Musk leads Tesla")
print("=" * 40)
predict_sentence(model, ["Elon", "Musk", "leads", "Tesla"], word2idx, idx2tag)

---
## 11. Inspect the Learned Transition Matrix

The transition matrix is one of the most interesting things to inspect. We expect the model to have learned:
- `B-PER → I-PER` : **high** score (valid continuation)
- `B-PER → I-LOC` : **low** score (type mismatch)
- `O → I-PER`     : **low** score (can't start I- without B-)
- `START → I-PER` : **very low** (hard-blocked)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

T = model.crf.transitions.detach().cpu().numpy()

# Clip extreme hard-constraint values for better visualisation
T_vis = np.clip(T, -20, 20)

tag_labels = [idx2tag[i] for i in range(NUM_TAGS)]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(T_vis, cmap='RdYlGn', aspect='auto')
plt.colorbar(im, ax=ax, label='Transition score (clipped to [-20, 20])')

ax.set_xticks(range(NUM_TAGS)); ax.set_xticklabels(tag_labels, rotation=45, ha='right')
ax.set_yticks(range(NUM_TAGS)); ax.set_yticklabels(tag_labels)
ax.set_xlabel('TO tag')
ax.set_ylabel('FROM tag')
ax.set_title('CRF Learned Transition Matrix\n(green = high, red = low/forbidden)')

# Annotate each cell
for i in range(NUM_TAGS):
    for j in range(NUM_TAGS):
        ax.text(j, i, f'{T_vis[i,j]:.1f}', ha='center', va='center', fontsize=7,
                color='black')

plt.tight_layout()
plt.show()

---
## 12. Step-by-step Viterbi Trace

Let's manually trace the Viterbi algorithm on a single sentence so you can see the exact numbers.

In [ ]:
def viterbi_trace(model, sentence: List[str], word2idx, idx2tag):
    """
    Trace the Viterbi DP table — print δ (best score so far) at each step.
    """
    model.eval()
    indices = [word2idx.get(tok, word2idx[UNK]) for tok in sentence]
    x  = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    ln = torch.tensor([len(indices)]).to(device)

    with torch.no_grad():
        emissions = model._get_emissions(x, ln)[0]  # (T, K)

    crf = model.crf
    T_len, K = emissions.size()
    backpointers = []

    # Initialise
    delta = crf.transitions[crf.start_idx, :] + emissions[0]

    tag_names = [idx2tag[i] for i in range(K)]

    print(f"{'':20s}" + "".join(f"{t:>10s}" for t in tag_names))
    print("-" * (20 + 10 * K))

    vals = [f"{v:.2f}" for v in delta.tolist()]
    print(f"{sentence[0] + ' (init)':<20s}" + "".join(f"{v:>10s}" for v in vals))

    for t in range(1, T_len):
        delta_expanded = delta.unsqueeze(1) + crf.transitions
        best_scores, best_prev = delta_expanded.max(dim=0)
        backpointers.append(best_prev)
        delta = best_scores + emissions[t]
        vals = [f"{v:.2f}" for v in delta.tolist()]
        print(f"{sentence[t]:<20s}" + "".join(f"{v:>10s}" for v in vals))

    terminal = delta + crf.transitions[:, crf.end_idx]
    _, best_last = terminal.max(dim=0)
    path = [best_last.item()]
    for bp in reversed(backpointers):
        path.append(bp[path[-1]].item())
    path.reverse()

    print()
    print("Best path:", [idx2tag[p] for p in path])


viterbi_trace(model, ["Barack", "Obama", "visited", "Berlin"], word2idx, idx2tag)

---
## 13. Summary — Key Concepts

| Concept | What it does | Where in code |
|---|---|---|
| **BiLSTM** | Captures context from both directions | `BiLSTMCRF._get_emissions` |
| **Emission scores** | Per-token tag scores from BiLSTM+Linear | `hidden2tag` layer |
| **Transition matrix** | Learned tag-to-tag compatibility | `CRF.transitions` |
| **Sequence score** | Sum of emissions + transitions | `CRF._score_sentence` |
| **Forward algorithm** | Compute log Z in O(T·K²) via DP + logsumexp | `CRF._forward_alg` |
| **NLL loss** | log Z − score(gold) | `CRF.neg_log_likelihood` |
| **Viterbi** | Best path via DP + max + backpointers | `CRF.viterbi_decode` |
| **Gradient clipping** | Stabilise RNN training | `clip_grad_norm_` |

### Why log-space?
Working with log-probabilities prevents floating-point underflow. The trick:
```
log Σ exp(a_i) = max(a) + log Σ exp(a_i − max(a))
```
ensures the exp() arguments are ≤ 0, so no overflow occurs.

### Forward vs Viterbi — one-line difference
```python
# Forward (marginal, for Z):
alpha_new = log_sum_exp(alpha + T, dim=0) + emission

# Viterbi (best path):
delta_new = max(delta + T, dim=0) + emission
             └── also save argmax as back-pointer
```

Both are O(T × K²) — efficient enough for real-world tag sets.